# Atrasando a entrada


1. Gerar modelo GPR 
2. Gerar posição das amostras virtuais

- 3 entradas:
    - Wd
    - We
    - thetaLag <- theta(Wd,We) deslocado 1 time step para trás
- 1 saída:
    - theta(Wd,We)


#  Gerando Amostras Virtuais

In [142]:
# === IMPORTS ===
import numpy as np
import pandas as pd
from scipy.spatial import Delaunay
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.offline as py
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, ExpSineSquared, DotProduct, WhiteKernel, ConstantKernel as C
)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error, root_mean_squared_error
from scipy.interpolate import UnivariateSpline


# Visualização

In [143]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def PrintInputSpace(original_samples_x, virtual_samples_x, time):

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.15,
    )

    # --- wd vs tempo ---
    fig.add_trace(go.Scatter(
        x=time,
        y=original_samples_x[:, 0],
        mode='markers',
        name='Entradas Originais',
        marker=dict(color='gray', size=8, opacity=0.8, symbol='circle')
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=time,
        y=virtual_samples_x[:, 0],
        mode='markers',
        name='Entradas Virtuais',
        marker=dict(color='blue', size=8, opacity=0.8, symbol='circle')
    ), row=1, col=1)

    # --- we vs tempo ---
    fig.add_trace(go.Scatter(
        x=time,
        y=original_samples_x[:, 1],
        mode='markers',
        name='Entradas Originais',
        marker=dict(color='gray', size=8, opacity=0.8, symbol='circle'),
        showlegend=False  # evita duplicar legenda
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=time,
        y=virtual_samples_x[:, 1],
        mode='markers',
        name='Entradas Virtuais',
        marker=dict(color='blue', size=8, opacity=0.8, symbol='circle'),
        showlegend=False
    ), row=2, col=1)

    fig.update_layout(
        font=dict(family="Times New Roman", size=20, color="black"),
        xaxis2=dict(title="Tempo [s]", showgrid=True, gridcolor='lightgray'),
        yaxis=dict(title=r"$\omega_d$", showgrid=True, gridcolor='lightgray'),
        yaxis2=dict(title=r"$\omega_e$", showgrid=True, gridcolor='lightgray'),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.2,
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgray',
            borderwidth=1
        ),
        template='plotly_white',
        width=700,
        height=600,
        margin=dict(t=100, b=100)
    )

    # Exporta para PDF
    output_path = f"./content/figs/InputSpace.pdf"
    fig.write_image(output_path, format='pdf')
    print(f"Figura salva em: {output_path}")


# Definições

In [144]:
PREDICTORS = ["Wd", "We", "thetaLag"]
TARGET = "theta(Wd,We)"

KERNELS = [
    ("RBF",                   C(1.0) * RBF(length_scale=1.0, length_scale_bounds=(1e-6, 1e+6))),
    ("Matern_0.5",            C(1.0) * Matern(length_scale=1.0, nu=0.5, length_scale_bounds=(1e-6, 1e+6))),
    ("RationalQuadratic",     C(1.0) * RationalQuadratic(length_scale=1.0, alpha=0.1, length_scale_bounds=(1e-6, 1e+6))),
    ("DotProduct",            C(1.0) * DotProduct() + WhiteKernel()),
    ("ExpSineSquared",        C(1.0) * ExpSineSquared(length_scale=1.0, periodicity=3.0, length_scale_bounds=(1e-6, 1e+6)))

]

TRAINFILES = ["DataSpline1"]
TESTFILES = ["DataSpline2", "DataSpline3", "DataSpline4"]

SPLINEPARAMS = {
    "Wd": (1, 0.06),
    "We": (1, 0.06),
    "thetaLag": (3, 0.06),
}

# Preparação dos dados

In [145]:
def GetTrainData(file):
    TrainData = pd.read_csv(f"../Dados/{file}.csv")

    # renomeia a primeira coluna para "time"
    first_col = TrainData.columns[0]
    TrainData = TrainData.rename(columns={first_col: "time"})

    # define "time" como índice
    TrainData = TrainData.set_index("time")

    TrainData["thetaLag"] = TrainData["theta(Wd,We)"].shift(1)
    TrainData = TrainData.iloc[1:]
    return TrainData


def GetTestData(file):
    TestData = pd.read_csv(f"../Dados/{file}.csv")

    # renomeia a primeira coluna para "time"
    first_col = TestData.columns[0]
    TestData = TestData.rename(columns={first_col: "time"})

    # define "time" como índice
    TestData = TestData.set_index("time")

    TestData["thetaLag"] = TestData["theta(Wd,We)"].shift(1)
    TestData = TestData.iloc[1:]

    test_samples_x = np.array(TestData[PREDICTORS])
    test_samples_y = np.array(TestData[TARGET])
    test_time = TestData.index
    return test_samples_x, test_samples_y, test_time


# Geração das entradas virtuais

In [146]:
def SetInputSpace(TrainData, trainTime, periodo=None, plot=False):
    data_spline = {}
    virtual_samples = {}

    # amostras geradas no centro entre duas amostras reais
    virtual_time = (trainTime[:-1] + trainTime[1:]) / 2

    for predictor, (k, fator_s) in SPLINEPARAMS.items():
        y = TrainData[predictor].to_numpy()
        s_val = len(trainTime) * np.var(y) * fator_s

        # spline ajustada
        spline = UnivariateSpline(trainTime, y, s=s_val, k=k)

        # avalia nos instantes originais e nos virtuais
        data_spline[predictor] = spline(trainTime)
        virtual_samples[predictor] = spline(virtual_time)

        if plot:
            plt.figure(figsize=(10, 4))
            plt.plot(trainTime, y, 'o', label='Dados Reais')
            plt.plot(virtual_time, virtual_samples[predictor], 'x', 
                    label=f"Entradas Virtuais (k={k}, s={fator_s})")
            plt.title(f"Amostras reais e virtuais para {predictor}")
            plt.xlabel("Tempo (s)")
            plt.ylabel(predictor)
            plt.grid(True)
            plt.legend()
            plt.tight_layout()

            # se foi passado um período, limita o eixo x
            if periodo is not None:
                plt.xlim(trainTime.min(), trainTime.min() + periodo)

        plt.show()

    # salva entradas virtuais em planilha
    df_virtual = pd.DataFrame(virtual_samples, index=virtual_time)
    return df_virtual


# Geração das saidas virtuais

In [147]:
class GprVsg3D:
    def __init__(self):        
        TrainData = GetTrainData(TRAINFILES[0])
        self.train_samples_x = np.array(TrainData[PREDICTORS])
        self.train_samples_y = np.array(TrainData[TARGET])
        self.train_time = TrainData.index
        self.model = None
        
        self.d = self.train_samples_x.shape[1]

        VirtualData = SetInputSpace(TrainData, self.train_time, periodo=4)
  
        self.virtual_samples_x = np.array(VirtualData[PREDICTORS])

        PrintInputSpace(self.train_samples_x, self.virtual_samples_x, self.train_time)

        self.virtual_samples_y = []

    def BuildModel(self, kernel):
        self.model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2,
                                              alpha=1e-7, normalize_y=True)
        self.model.fit(self.train_samples_x, self.train_samples_y)
    
    def Evaluate(self, kernel_name):
        colors = plt.cm.tab10.colors  

        # subplots empilhados (treino + n testes)
        fig, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=False)
        axes = axes.flatten()   # transforma em array 1D com 4 posições

        # === Previsão treino ===
        self.train_pred = self.model.predict(self.train_samples_x)
        
        metrics = {
            'r2_training': r2_score(self.train_samples_y, self.train_pred),
            'mse_training': mean_squared_error(self.train_samples_y, self.train_pred),
            'mape_training': mean_absolute_percentage_error(self.train_samples_y, self.train_pred),
            'rmse_training': root_mean_squared_error(self.train_samples_y, self.train_pred)
        }
        
        ax = axes[0]
        ax.plot(self.train_time, self.train_pred, marker="o", linewidth=2,
                label="train_pred", color=colors[0])
        ax.plot(self.train_time, self.train_samples_y, marker="o", linewidth=2,
                label="train_orig", color=colors[1])
        ax.set_title("Dados de Treinamento", fontsize=13)
        ax.set_xlabel("Tempo (s)", fontsize=12)
        ax.set_ylabel("Velocidade Angular", fontsize=12)
        ax.legend()
        ax.grid(True)
        
        # === Loop nos testes ===
        for i, file in enumerate(TESTFILES):
            test_samples_x, test_samples_y, test_time = GetTestData(file)
            test_pred = self.model.predict(test_samples_x)
            
            metrics.update({
                f'r2_test{i+1}': r2_score(test_samples_y, test_pred),
                f'mse_test{i+1}': mean_squared_error(test_samples_y, test_pred),
                f'mape_test{i+1}': mean_absolute_percentage_error(test_samples_y, test_pred),
                f'rmse_test{i+1}': root_mean_squared_error(test_samples_y, test_pred)
            })
            
            ax = axes[i+1]  # subplots [1], [2], [3] para os testes
            ax.plot(test_time, test_pred, marker="o", linewidth=2,
                    label=f"test_pred_{i+1}", color=colors[2*i % len(colors)])
            ax.plot(test_time, test_samples_y, marker="o", linewidth=2,
                    label=f"test_orig_{i+1}", color=colors[(2*i+1) % len(colors)])
            ax.set_title(f"Dados de Teste {i+1} ({file})", fontsize=13)
            ax.set_xlabel("Tempo (s)", fontsize=12)
            ax.set_ylabel("Velocidade Angular", fontsize=12)
            ax.legend()
            ax.grid(True)

        # Título geral
        fig.suptitle(f"Resultados do modelo GPR: {kernel_name}", fontsize=15)
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(f"content/figs/{kernel_name}.pdf", format="pdf", dpi=600)
        plt.show()
        
        return metrics 
                 
    def EvalModel(self, excel_path):
        results = []  # lista para armazenar todas as métricas
        
        for name, kernel in KERNELS:
            self.BuildModel(kernel=kernel)
            metrics = self.Evaluate(kernel_name=name)
            display(metrics)
            metrics["Kernel"] = name  # adiciona o nome do kernel como coluna
            results.append(metrics)

        # cria um único DataFrame com todos os resultados
        df = pd.DataFrame(results)

        # reorganiza para que "Kernel" fique como primeira coluna
        cols = ["Kernel"] + [c for c in df.columns if c != "Kernel"]
        df = df[cols]

        # salva em uma única planilha
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="Resultados", index=False)
            

    def ComputeY(self):
        self.virtual_samples_y = self.model.predict(self.virtual_samples_x)
        self.virtual_samples_df = pd.DataFrame({
            "Wd": self.virtual_samples_x[:, 0],
            "We": self.virtual_samples_x[:, 1],
            "theta(Wd,We)": self.virtual_samples_y
        })
        return self.virtual_samples_df
    
    
    def GetVirtualSamples(self):
        for name, kernel in KERNELS:
            self.BuildModel(kernel=kernel)
            self.ComputeY()
            
            self.virtual_samples_df.to_excel(f"./content/VirtualSamples/virtual_samples_{name}.xlsx", index=False)

In [148]:
Vsg = GprVsg3D()

Figura salva em: ./content/figs/InputSpace.pdf


In [149]:
#Vsg.GetVirtualSamples()

In [150]:
# Vsg.EvalModel("./content/GprMetrics.xlsx")